# match_startups_to_patents

**Objective:** Link startups to Lens patent applicants through a four-tier name-matching cascade (exact with and without country, then fuzzy at two thresholds), then explode the links to one row per startup-patent pair.

**Inputs:** `../02_matching_crunchbase_data/startups_master.csv`; Lens JSONL exports in `../03_lens_query_and_ingestion/raw_batches/`.

**Outputs:** `match_outputs/startup_applicant_matches.csv`, `startup_patent_matches.csv`, and `unmatched_startups.csv`.

## Imports

In [1]:
# Imports
import json, gzip, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from unidecode import unidecode

## Paths and fuzzy thresholds

In [2]:
# Paths and fuzzy thresholds
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
STARTUP_CSV = PROC / "startups_master.csv"
JSONL_DIR   = RAW / "lens"
OUT_DIR     = PROC

FUZZY_TIER_C_THRESHOLD = 90
FUZZY_TIER_D_THRESHOLD = 95

## 1. Load startups and map countries to ISO2

In [3]:
# Load startups and map their headquarters country to ISO-2 codes
startups = pd.read_csv(STARTUP_CSV)
print(f"Loaded {len(startups):,} startups")

COUNTRY_TO_ISO2 = {
    "Austria": "AT", "Belgium": "BE", "Bulgaria": "BG", "Croatia": "HR",
    "Cyprus": "CY", "Czech Republic": "CZ", "Czechia": "CZ", "Denmark": "DK",
    "Estonia": "EE", "Finland": "FI", "France": "FR", "Germany": "DE",
    "Greece": "GR", "Hungary": "HU", "Iceland": "IS", "Ireland": "IE",
    "Italy": "IT", "Latvia": "LV", "Liechtenstein": "LI", "Lithuania": "LT",
    "Luxembourg": "LU", "Malta": "MT", "The Netherlands": "NL", "Netherlands": "NL",
    "Norway": "NO", "Poland": "PL", "Portugal": "PT", "Romania": "RO",
    "Slovakia": "SK", "Slovenia": "SI", "Spain": "ES", "Sweden": "SE",
    "Switzerland": "CH", "United Kingdom": "GB", "UK": "GB",
    "Russian Federation": "RU", "Russia": "RU", "Turkey": "TR", "Ukraine": "UA",
    "Belarus": "BY", "Serbia": "RS", "Bosnia and Herzegovina": "BA",
    "Macedonia (FYROM)": "MK", "North Macedonia": "MK", "Albania": "AL",
    "Montenegro": "ME", "Moldova": "MD", "Monaco": "MC", "Andorra": "AD",
    "Faroe Islands": "FO", "Greenland": "GL", "San Marino": "SM",
    "Vatican City": "VA", "Gibraltar": "GI", "Slovakia (Slovak Republic)": "SK",
    "Isle of Man": "IM",
}
startups["country_iso2"] = startups["country"].map(COUNTRY_TO_ISO2)
print(f"Mapped {startups['country_iso2'].notna().sum():,} / {len(startups):,} to ISO2")

unmapped = startups.loc[startups["country_iso2"].isna(), "country"].value_counts()
if len(unmapped):
    print("Unmapped countries:")
    print(unmapped.head(15))

Loaded 6,642 startups
Mapped 6,642 / 6,642 to ISO2


## 2. Load JSONL applicants

In [4]:
# Load Lens JSONL applicants into a flat applicant table
def open_any(p):
    return gzip.open(p, "rb") if p.name.endswith(".gz") else open(p, "rb")

def iter_jsonl(path):
    with open_any(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line.decode("utf-8-sig"))
            except Exception:
                continue

seen_lens_ids = set()
applicant_rows = []
jsonl_files = sorted(JSONL_DIR.glob("*.jsonl*"))
print(f"\nProcessing {len(jsonl_files)} JSONL files...")

for f in jsonl_files:
    for d in iter_jsonl(f):
        lid = d.get("lens_id")
        if not lid or lid in seen_lens_ids:
            continue
        seen_lens_ids.add(lid)

        biblio = d.get("biblio", {}) or {}
        parties = biblio.get("parties", {}) or {}
        applicants = parties.get("applicants", []) or []

        prio_dates = [
            pc.get("date")
            for pc in (biblio.get("priority_claims", {}) or {}).get("claims", []) or []
            if pc.get("date")
        ]
        earliest_prio = min(prio_dates) if prio_dates else None

        ipc = (biblio.get("classifications_ipcr", {}) or {}).get("classifications", []) or []
        primary_ipc4 = ipc[0]["symbol"][:4] if ipc and ipc[0].get("symbol") else None

        for app in applicants:
            ext = app.get("extracted_name") or {}
            name = ext.get("value") if isinstance(ext, dict) else app.get("name")
            applicant_rows.append({
                "lens_id": lid,
                "doc_key": d.get("doc_key"),
                "jurisdiction": d.get("jurisdiction"),
                "applicant_raw": name,
                "applicant_country": app.get("residence"),
                "earliest_priority": earliest_prio,
                "primary_ipc4": primary_ipc4,
            })

applicants_df = pd.DataFrame(applicant_rows)
print(f"Unique patents: {applicants_df['lens_id'].nunique():,}")
print(f"Applicant rows: {len(applicants_df):,}")
print(f"Distinct applicant names: {applicants_df['applicant_raw'].nunique():,}")


Processing 87 JSONL files...
Unique patents: 122,540
Applicant rows: 194,728
Distinct applicant names: 60,830


## 3. Name normalisation

In [ ]:
# Normalise company names (transliterate, strip punctuation and legal/generic tokens)
LEGAL_FORMS = {
    "gmbh", "mbh", "ag", "kg", "ohg", "kgaa", "se", "ug", "ev",
    "ltd", "limited", "llc", "lp", "llp", "plc", "inc", "incorporated",
    "corp", "corporation", "co", "company",
    "sa", "sas", "sasu", "sarl", "snc", "scs", "sca", "scop", "sci", "eurl", "soc",
    "societe", "compagnie", "cie",
    "srl", "spa", "sl", "slu", "scl", "lda", "unipersonal",
    "ab", "oy", "oyj", "as", "asa", "aps", "kommanditbolag", "hb",
    "bv", "nv", "cv", "vof", "be",
    "ltda", "anpartsselskab", "aktiebolag",
}
GENERIC_FILLERS = {
    "group", "groupe", "groep", "gruppe", "grupo",
    "holding", "holdings", "holdingen",
    "international", "intl", "global", "europe", "european", "world", "worldwide",
    "technologies", "technology", "tech", "techn", "techno", "technique", "techniques",
    "systems", "system", "solutions", "solution", "services", "service",
    "industries", "industry", "industrial",
    "innovations", "innovation", "innovative",
    "ventures", "venture", "capital",
    "research", "rech", "recherches",
    "development", "developments", "develop", "dev",
    "the", "le", "la", "les", "die", "der", "das", "il", "lo", "el", "los",
    "of", "and", "et", "und", "y", "&",
}
LEGAL_FORMS_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(t) for t in sorted(LEGAL_FORMS, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)
GENERIC_FILLERS_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(t) for t in sorted(GENERIC_FILLERS, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)
PUNCT_RE = re.compile(r"[^a-z0-9 ]+")
WS_RE = re.compile(r"\s+")

def normalise_name(name):
    if not isinstance(name, str) or not name.strip():
        return None
    s = unidecode(name).lower()
    s = PUNCT_RE.sub(" ", s)
    s = LEGAL_FORMS_RE.sub(" ", s)
    s = GENERIC_FILLERS_RE.sub(" ", s)
    s = WS_RE.sub(" ", s).strip()
    tokens = [t for t in s.split() if len(t) >= 2]
    if not tokens:
        return None
    return " ".join(tokens)

startups["name_norm"] = startups["Organization Name"].map(normalise_name)
applicants_df["name_norm"] = applicants_df["applicant_raw"].map(normalise_name)

startups_clean   = startups[startups["name_norm"].notna()].copy()
applicants_clean = applicants_df[applicants_df["name_norm"].notna()].copy()
print(f"\nStartups   with name_norm: {len(startups_clean):,} / {len(startups):,}")
print(f"Applicants with name_norm: {len(applicants_clean):,} / {len(applicants_df):,}")

## 4. Stage A — exact match on (name_norm, country)

In [ ]:
# Tier A: exact match on normalised name and country
applicants_unique = (
    applicants_clean
    .groupby(["name_norm", "applicant_country"])
    .agg(
        applicant_raw_examples=("applicant_raw", lambda s: list(set(s))[:3]),
        n_patents=("lens_id", "nunique"),
        lens_ids=("lens_id", lambda s: sorted(set(s))),
    )
    .reset_index()
)
stageA = startups_clean.merge(
    applicants_unique,
    left_on=["name_norm", "country_iso2"],
    right_on=["name_norm", "applicant_country"],
    how="inner",
)
stageA["match_tier"] = "A"
stageA["match_score"] = 100.0
print(f"\nStage A (exact name + country): {len(stageA):,} pairs / {stageA['Organization Name'].nunique():,} startups")

## 5. Stage B — exact name match, ignore country

In [ ]:
# Tier B: exact name match, ignoring country
matched_A = set(stageA["Organization Name"])
remaining_B = startups_clean[~startups_clean["Organization Name"].isin(matched_A)]
applicants_by_name = (
    applicants_clean
    .groupby("name_norm")
    .agg(
        applicant_raw_examples=("applicant_raw", lambda s: list(set(s))[:3]),
        countries=("applicant_country", lambda s: sorted({c for c in s if pd.notna(c)})),
        n_patents=("lens_id", "nunique"),
        lens_ids=("lens_id", lambda s: sorted(set(s))),
    )
    .reset_index()
)
stageB = remaining_B.merge(applicants_by_name, on="name_norm", how="inner")
stageB["match_tier"] = "B"
stageB["match_score"] = 100.0
stageB["applicant_country"] = stageB["countries"].apply(lambda cs: cs[0] if cs else None)
print(f"Stage B (exact name, no country): {len(stageB):,} pairs / {stageB['Organization Name'].nunique():,} new startups")

## 6. Stage C — fuzzy + same country (>=90)

In [ ]:
# Tier C: fuzzy match (token-sort >= 90) within the same country
matched_AB = matched_A | set(stageB["Organization Name"])
remaining_C = startups_clean[~startups_clean["Organization Name"].isin(matched_AB)].copy()
remaining_C = remaining_C[remaining_C["country_iso2"].notna()]

applicants_by_country = defaultdict(list)
for _, r in applicants_unique.iterrows():
    if pd.notna(r["applicant_country"]):
        applicants_by_country[r["applicant_country"]].append(r["name_norm"])

stageC_rows = []
for _, s in remaining_C.iterrows():
    country = s["country_iso2"]
    candidates = applicants_by_country.get(country, [])
    if not candidates:
        continue
    matches = process.extract(
        s["name_norm"], candidates,
        scorer=fuzz.token_sort_ratio,
        limit=3, score_cutoff=FUZZY_TIER_C_THRESHOLD,
    )
    for cand_norm, score, _ in matches:
        stageC_rows.append({
            "Organization Name": s["Organization Name"],
            "country_iso2": country,
            "name_norm_startup": s["name_norm"],
            "name_norm": cand_norm,
            "match_score": score,
        })
stageC = pd.DataFrame(stageC_rows)
if len(stageC):
    stageC = stageC.merge(
        applicants_unique,
        left_on=["name_norm", "country_iso2"],
        right_on=["name_norm", "applicant_country"],
        how="left",
    )
    stageC = stageC.sort_values(["Organization Name", "match_score"], ascending=[True, False])
    stageC["match_tier"] = "C"
print(f"Stage C (fuzzy >={FUZZY_TIER_C_THRESHOLD} + country): {len(stageC):,} pairs")

## 7. Stage D — fuzzy without country (>=95)

In [ ]:
# Tier D: fuzzy match (token-sort >= 95), ignoring country
matched_so_far = matched_AB | (set(stageC["Organization Name"]) if len(stageC) else set())
remaining_D = startups_clean[~startups_clean["Organization Name"].isin(matched_so_far)].copy()

all_applicant_norms = applicants_by_name["name_norm"].tolist()
stageD_rows = []
for _, s in remaining_D.iterrows():
    matches = process.extract(
        s["name_norm"], all_applicant_norms,
        scorer=fuzz.token_sort_ratio,
        limit=3, score_cutoff=FUZZY_TIER_D_THRESHOLD,
    )
    for cand_norm, score, _ in matches:
        stageD_rows.append({
            "Organization Name": s["Organization Name"],
            "country_iso2": s["country_iso2"],
            "name_norm_startup": s["name_norm"],
            "name_norm": cand_norm,
            "match_score": score,
        })
stageD = pd.DataFrame(stageD_rows)
if len(stageD):
    stageD = stageD.merge(applicants_by_name, on="name_norm", how="left")
    stageD["match_tier"] = "D"
print(f"Stage D (fuzzy >={FUZZY_TIER_D_THRESHOLD}, no country): {len(stageD):,} pairs")

## 8. Combine + resolve fuzzy collisions

In [ ]:
# Combine all tiers and resolve fuzzy collisions
keep_cols = [
    "Organization Name", "country_iso2", "name_norm",
    "applicant_country", "applicant_raw_examples",
    "n_patents", "lens_ids", "match_tier", "match_score",
]
all_stages = []
for stage in [stageA, stageB, stageC, stageD]:
    if len(stage):
        for c in keep_cols:
            if c not in stage.columns:
                stage[c] = np.nan
        all_stages.append(stage[keep_cols])
matches = pd.concat(all_stages, ignore_index=True)

fuzzy = matches[matches["match_tier"].isin(["C","D"])].sort_values("match_score", ascending=False)
fuzzy = fuzzy.drop_duplicates(subset=["name_norm", "applicant_country"], keep="first")
exact = matches[~matches["match_tier"].isin(["C","D"])]
matches = pd.concat([exact, fuzzy], ignore_index=True)

print(f"\nTotal matches after collision resolution: {len(matches):,}")
print("Tier breakdown:")
print(matches["match_tier"].value_counts().sort_index().to_string())
print(f"\nUnique startups matched: {matches['Organization Name'].nunique():,} / {len(startups_clean):,} "
      f"({matches['Organization Name'].nunique()/len(startups_clean)*100:.1f}%)")

## 9. Explode to per-patent rows

In [ ]:
# Explode matches to one row per startup-patent link
matches_exploded = matches.explode("lens_ids").rename(columns={"lens_ids": "lens_id"})
matches_exploded = matches_exploded[matches_exploded["lens_id"].notna()]

patent_meta = applicants_clean[
    ["lens_id", "applicant_raw", "applicant_country", "earliest_priority", "primary_ipc4", "jurisdiction"]
].drop_duplicates(subset=["lens_id", "applicant_raw"])

final = matches_exploded.merge(patent_meta, on="lens_id", how="left", suffixes=("", "_patent"))
print(f"\nFinal startup-patent links: {len(final):,}")
print(f"  unique startups: {final['Organization Name'].nunique():,}")
print(f"  unique patents:  {final['lens_id'].nunique():,}")

## 10. Outputs

In [ ]:
# Write applicant matches, patent links, and unmatched startups
match_out = matches.copy()
match_out["applicant_raw_examples"] = match_out["applicant_raw_examples"].apply(
    lambda x: " | ".join(x) if isinstance(x, list) else x
)
match_out = match_out.drop(columns=["lens_ids"])
match_out.to_csv(OUT_DIR / "startup_applicant_matches.csv", index=False)

final_out = final.drop(columns=["lens_ids", "applicant_raw_examples"], errors="ignore")
final_out.to_csv(OUT_DIR / "startup_patent_matches.csv", index=False)

unmatched = startups_clean[~startups_clean["Organization Name"].isin(matches["Organization Name"])]
unmatched_cols = [c for c in ["Organization Name", "country_iso2", "name_norm", "Industries"] if c in unmatched.columns]
unmatched[unmatched_cols].to_csv(OUT_DIR / "unmatched_startups.csv", index=False)

print(f"\nWrote:")
print(f"  {OUT_DIR / 'startup_applicant_matches.csv'}  ({len(match_out):,} rows)")
print(f"  {OUT_DIR / 'startup_patent_matches.csv'}     ({len(final_out):,} rows)")
print(f"  {OUT_DIR / 'unmatched_startups.csv'}         ({len(unmatched):,} rows)")